# StayIQ — Module 2: Review Sentiment & Aspect Analysis

This notebook covers the NLP pipeline for extracting granular guest sentiment. It combines deep learning sentiment classifiers (DistilBERT) with aspect-based keyword rule association, allowing hotel managers to trace *why* reviews are positive or negative.

### Pipeline Steps:
1. **Load reviews**: Read dataset schema containing guest review comments.
2. **Preprocess Texts**: Clean reviews, tokenize, and split reviews into sentence chunks.
3. **Transformer Sentiment Analysis**: Load a pre-trained DistilBERT pipeline for overall sentiment scoring.
4. **Aspect Extraction**: Scan sentences for aspect keywords (`cleanliness`, `staff`, `food`, `location`, `value`).
5. **Aspect-Level Sentiment Attribution**: Evaluate sentiment scores on aspect-specific sentence subsets.
6. **Export pipeline notes**: Save local aspect scoring details.

In [ ]:
# 1. Imports and setup
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Check for transformers availability
try:
    from transformers import pipeline
    print("HuggingFace Transformers imported successfully.")
except ImportError:
    print("HuggingFace Transformers is not installed. Run 'pip install transformers torch' to activate DistilBERT components.")

In [ ]:
# 2. Load Sample Review Data
# Note: Place your reviews CSV in data/
reviews = [
    "The room was very clean and neat, but reception staff was a bit slow and rude.",
    "Excellent dining at the restaurant! Delicious buffet breakfast. Worth every penny.",
    "Perfect central location, walking distance to all sites. The bathroom was dusty though.",
    "Overpriced hotel. Small rooms, and the breakfast meal was terrible and cold.",
    "Extremely friendly service from start to finish. The view was amazing. Highly recommend!"
]
df = pd.DataFrame({'review_text': reviews})
df

In [ ]:
# 3. Sentiment Classifier (DistilBERT)
try:
    # Load pre-trained sentiment analysis model (DistilBERT-SST-2)
    nlp_classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")
    
    # Test classification
    for text in df['review_text']:
        res = nlp_classifier(text)[0]
        print(f"Review: '{text}'")
        print(f"-> Prediction: {res['label']} (Confidence: {res['score']:.4f})\n")
except Exception as e:
    print(f"Could not run HuggingFace pipeline: {e}. Standard library simulation is used in backend fallback.")

In [ ]:
# 4. Aspect-Based Keyword Extractor

ASPECT_MAP = {
    'cleanliness': ['clean', 'dusty', 'dirty', 'spotless', 'bathroom', 'shower', 'sheets', 'hygiene'],
    'staff': ['staff', 'reception', 'clerk', 'service', 'friendly', 'rude', 'host', 'manager'],
    'food': ['food', 'breakfast', 'buffet', 'restaurant', 'meal', 'delicious', 'cold', 'dining'],
    'location': ['location', 'distance', 'metro', 'walk', 'beach', 'views', 'central', 'close'],
    'value': ['price', 'value', 'money', 'expensive', 'cheap', 'overpriced', 'cost', 'worth']
}

def extract_aspect_sentiment(review_text):
    text = review_text.lower()
    # Split sentences
    sentences = re.split(r'[.!?]+', text)
    
    extracted_aspects = {}
    for aspect, keywords in ASPECT_MAP.items():
        matches = []
        for sent in sentences:
            sent = sent.strip()
            if not sent:
                continue
            # Check word match
            if any(re.search(r'\b' + re.escape(kw) + r'\b', sent) for kw in keywords):
                matches.append(sent)
        
        if matches:
            extracted_aspects[aspect] = matches
            
    return extracted_aspects

# Test aspect extractor
for r in df['review_text']:
    print(f"Review: '{r}'")
    found = extract_aspect_sentiment(r)
    for asp, sentences in found.items():
        print(f"  - Aspect [{asp}]: Matches sentence -> \"{sentences}\"")
    print()

### Integration Summary:
In backend deployment, the transformer model weights (approx 260MB) are too heavy for Vercel Serverless's 250MB limit. Hence, the code is structured to lazy-load. In Vercel environments, we default to the fast, rule-based keywords-sentiment engine, while the standalone Render/Railway deployment runs this full Transformer pipeline.